### Regression Model 

In [11]:
import pandas as pd
import numpy as np
from configs.system import project_dir
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [13]:
p_dir = project_dir()
dataset_path = p_dir.FEATURED_DIR/'featured_dataset.csv'

df = pd.read_csv(dataset_path,index_col=0)
df.sample(2)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,StartMonth,StartWeek,EndMonth,EndWeek,DemandPerMonth,StartDayOfWeek,DemandPerDay
12382,64,64,1,apartment,0,2021-10-15,06:07:38,2021-10-15,07:00:35,2021-10-15 06:07:00,2021-10-15 07:00:00,53,6.10,10,4,10,4,64933.48,Friday,17.31
28295,0,211,1,company,0,2022-07-31,06:36:55,2022-07-31,07:23:06,2022-07-31 06:36:00,2022-07-31 07:23:00,46,5.32,7,6,7,6,139686.65,Sunday,17.81


In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 72826 entries, 0 to 72855
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          72826 non-null  int64  
 1   ChargerID       72826 non-null  int64  
 2   ChargerCompany  72826 non-null  int64  
 3   Location        72826 non-null  str    
 4   ChargerType     72826 non-null  int64  
 5   StartDay        72826 non-null  str    
 6   StartTime       72826 non-null  str    
 7   EndDay          72826 non-null  str    
 8   EndTime         72826 non-null  str    
 9   StartDatetime   72826 non-null  str    
 10  EndDatetime     72826 non-null  str    
 11  Duration        72826 non-null  int64  
 12  Demand          72826 non-null  float64
 13  StartMonth      72826 non-null  int64  
 14  StartWeek       72826 non-null  int64  
 15  EndMonth        72826 non-null  int64  
 16  EndWeek         72826 non-null  int64  
 17  DemandPerMonth  72826 non-null  float64
 18  St

In [16]:
df.sample()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,StartMonth,StartWeek,EndMonth,EndWeek,DemandPerMonth,StartDayOfWeek,DemandPerDay
72817,1869,2609,0,apartment,0,2022-01-25,19:37:05,2022-01-26,01:00:28,2022-01-25 19:37:00,2022-01-26 01:00:00,323,12.64,1,1,1,2,109639.33,Tuesday,17.1


In [ ]:
categorical_columns = ['Location','StartDayOfWeek']
numerical_col = ['ChargerCompany','ChargerType','Duration','StartMonth','StartWeek','DemandPerMonth','DemandPerDay']

In [26]:
X = df[['Location','StartDayOfWeek','ChargerCompany','ChargerType','Duration','StartMonth','StartWeek','DemandPerMonth','DemandPerDay']]
y = df['Demand']

In [27]:
X

,Location,StartDayOfWeek,ChargerCompany,ChargerType,Duration,StartMonth,StartWeek,DemandPerMonth,DemandPerDay
0,hotel,Thursday,1,0,185,9,3,111771.80,17.62
1,hotel,Wednesday,1,0,90,9,2,111771.80,17.53
2,hotel,Wednesday,1,0,60,9,2,111771.80,17.53
3,hotel,Thursday,1,0,344,9,3,111771.80,17.62
4,hotel,Sunday,1,0,300,9,6,111771.80,17.81
...,...,...,...,...,...,...,...,...,...
72851,public institution,Thursday,0,0,4,1,3,109639.33,17.62
72852,sightseeing,Sunday,0,1,28,10,6,64933.48,17.81
72853,sightseeing,Sunday,0,1,40,10,6,64933.48,17.81
72854,company,Thursday,0,1,8,11,3,71741.55,17.62


In [30]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [28]:
ln = LinearRegression()

In [54]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_columns = [
    'Location','StartDayOfWeek'
]
numerical_columns = [
    'ChargerCompany','ChargerType','Duration','StartMonth','StartWeek'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_columns),
        ('num','passthrough',numerical_columns)
    ]
)


model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('regressor',LinearRegression())
    ]
)



In [55]:
model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['Location','StartDayOfWeek','ChargerCompany',...,'StartWeek', 'DemandPerMonth','DemandPerDay']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed throu

In [56]:
y_pred = model.predict(X_test)

In [58]:
r2_score_ln = r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
mse = mean_squared_error(y_test,y_pred)

print('Linear Regression Score')
print(r2_score_ln)
print(mae)
print(mse)


Linear Regression Score
0.5676364109232173
5.623351960962902
77.91109028666288


### Decision Tree Regression

In [62]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [64]:

depths = [1,2,3,4,5,6,7,8,9,10]

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth,random_state=42)

    dt.fit(X_train,y_train)


ValueError: could not convert string to float: 'hotel'